# 学習候補データの評価値探索・選定ワークベンチ（WD14 + rtmlib + DINOv2）

複数パターンで生成した LoRA 学習候補画像群（サブフォルダ＝パターン単位）の評価値を探索し、**実際の学習データ選定基準の策定〜フィルタリング〜プロファイル生成**までを一貫して行うノートブックです。

3つのモデル軸で全候補画像を数値化します。

1. **WD14-Tagger（`SmilingWolf/wd-eva02-large-tagger-v3`）**: タグ信頼度スコア（属性・表情・視線・構図・品質・外乱タグの連続値）
2. **rtmlib WholeBody（COCO WholeBody 133点）**: 骨格検出品質・画角・向き・スケール不変プロポーション（頭身比等）・関節角度
3. **Hugging Face DINOv2（`facebook/dinov2-base`）**: 画像埋め込みコサイン類似度による絵柄・顔つきの一貫性（全体/パターン重心との類似度）および似すぎ画像の重複検知

### ワークフロー
1. **セットアップ & データ準備**: パターン別サブフォルダ（`./data/candidates/<pattern>/`）の読み込み
2. **一括計測**: 全候補画像に対して WD14 + rtmlib + DINOv2 を実行し、評価値テーブルを構築
3. **評価値の探索（Explore）**: 分布グラフ・タグ安定度マップ・任意タグのパターン別比較・極端値サムネイル確認から、しきい値と選定基準を探る
4. **選定ルールの適用（Curate）**: 条件・重複除外ルールを試行錯誤し、採用/除外の目視確認とパターン別バランス確認を行う
5. **書き出し & プロファイル生成**: 採用データセットの書き出しと、`snapshot-inspector` で使える `character_profile.json` の生成

## 1. セットアップ

必要なライブラリ（`rtmlib`, `onnxruntime-gpu`, `opencv-python`, `huggingface_hub`, `transformers`, `torch`, `pillow`, `pandas`, `matplotlib`）をインストールし、GPUランタイムの状態を確認します。

In [ ]:
%pip install -q rtmlib onnxruntime-gpu opencv-python huggingface_hub transformers pillow pandas matplotlib

In [ ]:
!nvidia-smi

## 2. Google Drive マウント & 学習候補データ準備（サブフォルダ＝パターン対応）

`./data/candidates/` 配下にパターンごとのサブフォルダを作成して画像を配置します（直下に置いた画像も `pattern="root"` として自動認識されます）。

```text
./data/candidates/
  ├── pattern_a_front/
  │   ├── 0001.png
  │   └── ...
  ├── pattern_b_side/
  └── pattern_c_fullbody/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
CANDIDATE_ROOT = "./data/candidates"
!mkdir -p {CANDIDATE_ROOT}

# Google Drive からパターンごとのフォルダを丸ごとコピーする例（環境に合わせてコメント解除・編集してください）
# !cp -r '/content/drive/MyDrive/Colab Notebooks/lora-candidates/'* {CANDIDATE_ROOT}/

# 配置確認
!find {CANDIDATE_ROOT} -maxdepth 2 -type d | sort

## 3. モデル初期化（rtmlib + WD14-Tagger v3 + Hugging Face DINOv2）

- **骨格検出**: `rtmlib.Wholebody`（COCO WholeBody 133点）
- **タグ信頼度**: `SmilingWolf/wd-eva02-large-tagger-v3`（`onnxruntime-gpu`）
- **視覚埋め込み**: `facebook/dinov2-base`（Hugging Face `transformers` + PyTorch GPU）

In [ ]:
import os
import json
import math
import shutil
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort
import torch
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import AutoImageProcessor, AutoModel
from rtmlib import Wholebody, draw_skeleton

# デバイス自動判定
AVAILABLE_PROVIDERS = ort.get_available_providers()
USE_CUDA_ORT = "CUDAExecutionProvider" in AVAILABLE_PROVIDERS
RTMLIB_DEVICE = "cuda" if USE_CUDA_ORT else "cpu"
ORT_PROVIDERS = (
    ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if USE_CUDA_ORT
    else ["CPUExecutionProvider"]
)
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"ONNX Runtime: {AVAILABLE_PROVIDERS} -> device='{RTMLIB_DEVICE}'")
print(f"PyTorch     : device='{TORCH_DEVICE}'")

# 1. rtmlib Wholebody
wholebody = Wholebody(
    to_openpose=False,
    mode="performance",
    backend="onnxruntime",
    device=RTMLIB_DEVICE,
)


# 2. WD14-Tagger (ONNX)
class WD14Tagger:
    """SmilingWolf 系 WD14-Tagger (ONNX) ラッパー。全タグの連続値信頼度スコアを返す。"""

    def __init__(self, repo_id="SmilingWolf/wd-eva02-large-tagger-v3", providers=None):
        self.repo_id = repo_id
        self.providers = providers or ORT_PROVIDERS

        model_path = hf_hub_download(repo_id=repo_id, filename="model.onnx")
        tags_path = hf_hub_download(repo_id=repo_id, filename="selected_tags.csv")

        self.session = ort.InferenceSession(model_path, providers=self.providers)
        self.input_meta = self.session.get_inputs()[0]
        self.input_name = self.input_meta.name
        self.output_name = self.session.get_outputs()[0].name
        self.target_size = int(self.input_meta.shape[1])

        tags_df = pd.read_csv(tags_path)
        self.tag_names = tags_df["name"].astype(str).tolist()
        self.tag_categories = tags_df["category"].astype(int).tolist()
        self.rating_indices = [i for i, c in enumerate(self.tag_categories) if c == 9]
        self.general_indices = [i for i, c in enumerate(self.tag_categories) if c == 0]
        self.character_indices = [i for i, c in enumerate(self.tag_categories) if c == 4]
        print(f"Loaded WD14Tagger: {repo_id} (size={self.target_size}, tags={len(self.tag_names)})")

    def preprocess(self, img_bgr: np.ndarray) -> np.ndarray:
        if img_bgr.ndim == 2:
            img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
        elif img_bgr.shape[2] == 4:
            alpha = img_bgr[:, :, 3:4].astype(np.float32) / 255.0
            bgr = img_bgr[:, :, :3].astype(np.float32)
            img_bgr = (bgr * alpha + 255.0 * (1.0 - alpha)).astype(np.uint8)

        h, w = img_bgr.shape[:2]
        max_dim = max(h, w)
        pad_top = (max_dim - h) // 2
        pad_bottom = max_dim - h - pad_top
        pad_left = (max_dim - w) // 2
        pad_right = max_dim - w - pad_left
        padded = cv2.copyMakeBorder(
            img_bgr,
            pad_top, pad_bottom, pad_left, pad_right,
            borderType=cv2.BORDER_CONSTANT,
            value=(255, 255, 255),
        )
        interp = cv2.INTER_AREA if max_dim > self.target_size else cv2.INTER_CUBIC
        resized = cv2.resize(padded, (self.target_size, self.target_size), interpolation=interp)
        return np.expand_dims(resized.astype(np.float32), axis=0)

    def predict(self, img_bgr: np.ndarray) -> dict:
        tensor = self.preprocess(img_bgr)
        probs = self.session.run([self.output_name], {self.input_name: tensor})[0][0]
        return {
            "rating": {self.tag_names[i]: float(probs[i]) for i in self.rating_indices},
            "general": {self.tag_names[i]: float(probs[i]) for i in self.general_indices},
            "character": {self.tag_names[i]: float(probs[i]) for i in self.character_indices},
        }


# 3. Hugging Face DINOv2 画像埋め込み抽出器
class DINOv2Embedder:
    """Hugging Face DINOv2 で L2正規化済みの画像特徴ベクトル（CLSトークン）を抽出する。"""

    def __init__(self, model_id="facebook/dinov2-base", device=None):
        self.model_id = model_id
        self.device = device or TORCH_DEVICE
        self.processor = AutoImageProcessor.from_pretrained(model_id)
        self.model = AutoModel.from_pretrained(model_id).to(self.device)
        self.model.eval()
        print(f"Loaded DINOv2Embedder: {model_id} on {self.device}")

    @torch.no_grad()
    def embed(self, img_bgr: np.ndarray) -> np.ndarray:
        if img_bgr.ndim == 2:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2RGB)
        elif img_bgr.shape[2] == 4:
            alpha = img_bgr[:, :, 3:4].astype(np.float32) / 255.0
            bgr = img_bgr[:, :, :3].astype(np.float32)
            composited = (bgr * alpha + 255.0 * (1.0 - alpha)).astype(np.uint8)
            img_rgb = cv2.cvtColor(composited, cv2.COLOR_BGR2RGB)
        else:
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        pil_img = Image.fromarray(img_rgb)
        inputs = self.processor(images=pil_img, return_tensors="pt").to(self.device)
        outputs = self.model(**inputs)
        # CLS トークンを L2 正規化してコサイン類似度計算にそのまま使える形式へ
        cls_emb = outputs.last_hidden_state[:, 0, :].float()
        cls_emb = torch.nn.functional.normalize(cls_emb, p=2, dim=-1)
        return cls_emb.squeeze(0).cpu().numpy()


WD14_MODEL_REPO = "SmilingWolf/wd-eva02-large-tagger-v3"
DINOV2_MODEL_ID = "facebook/dinov2-base"

wd14_tagger = WD14Tagger(repo_id=WD14_MODEL_REPO, providers=ORT_PROVIDERS)
dinov2_embedder = DINOv2Embedder(model_id=DINOV2_MODEL_ID, device=TORCH_DEVICE)

## 4. 計測・探索・選定ユーティリティ関数

以下のモジュール群を定義します。

- **骨格・タグ変数抽出**: `discover_candidate_images`, `extract_pose_variables`, `extract_tag_variables`, `analyze_candidates`
- **評価値の探索（Explore）**: `summarize_tag_landscape`, `show_exploration_overview`, `inspect_tag_across_patterns`, `show_extreme_thumbnails`
- **選定ルールの適用（Curate）**: `apply_selection_rules`, `show_selection_summary`, `show_curated_thumbnails`, `export_selected_dataset`, `build_profile_from_selected`

In [ ]:
# ==========================================
# スロット定義 & 骨格変数抽出関数
# ==========================================
VARIABLE_TAG_SLOTS = {
    "expression": [
        "smile", "grin", "open_mouth", "closed_mouth", "parted_lips",
        "blush", "expressionless", "serious", "angry", "sad",
        "surprised", "frown", "embarrassed", "nervous", "tears",
        "closed_eyes", "one_eye_closed", "half-closed_eyes", "wide-eyed",
    ],
    "gaze": [
        "looking_at_viewer", "looking_away", "looking_to_the_side",
        "looking_back", "looking_up", "looking_down",
        "facing_viewer", "facing_away", "profile",
        "from_side", "from_behind", "straight-on",
    ],
    "framing": [
        "portrait", "close-up", "upper_body", "cowboy_shot",
        "full_body", "wide_shot", "dutch_angle", "from_above", "from_below",
    ],
    "pose_action": [
        "standing", "sitting", "walking", "running", "jumping",
        "arms_up", "arm_up", "hand_on_hip", "hands_on_hips", "crossed_arms",
        "peace_sign", "waving", "holding", "leaning_forward", "head_tilt",
        "hands_behind_back", "arms_behind_back",
    ],
    "background": [
        "simple_background", "white_background", "grey_background",
        "black_background", "blue_background", "gradient_background",
        "transparent_background", "outdoors", "indoors", "sky",
    ],
}

IGNORED_IDENTITY_TAGS = {
    "1girl", "1boy", "solo", "solo_focus", "2girls", "2boys", "multiple_girls",
    "monochrome", "greyscale", "comic", "highres", "absurdres", "commentary_request",
    "shadow", "depth_of_field", "blurry", "blurry_background", "motion_blur",
    "light_rays", "lens_flare", "vignetting",
}

COCO_BODY_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]
KP = {name: idx for idx, name in enumerate(COCO_BODY_NAMES)}


def _round_or_none(val, digits=4):
    if val is None or (isinstance(val, float) and (math.isnan(val) or math.isinf(val))):
        return None
    return round(float(val), digits)


def discover_candidate_images(root_dir: str, extensions=(".png", ".jpg", ".jpeg", ".webp")):
    """
    サブフォルダを「生成パターン（pattern）」として再帰的に候補画像を収集する。
    ルート直下の画像は pattern='root' として扱う。
    """
    root = Path(root_dir)
    if not root.exists():
        return []

    items = []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue
        rel = path.relative_to(root)
        pattern = rel.parts[0] if len(rel.parts) > 1 else "root"
        image_id = f"{pattern}/{path.stem}" if pattern != "root" else path.stem
        items.append({
            "image_id": image_id,
            "pattern": pattern,
            "file_name": path.name,
            "file_path": str(path),
        })
    return items


def _select_primary_person(kpts, scores, kpt_thr=0.3):
    if kpts is None or len(kpts) == 0:
        return None, None
    if len(kpts) == 1:
        return np.asarray(kpts[0], dtype=np.float32), np.asarray(scores[0], dtype=np.float32)

    best_idx, best_weight = 0, -1.0
    for idx in range(len(kpts)):
        body_pts = np.asarray(kpts[idx][:17], dtype=np.float32)
        body_sc = np.asarray(scores[idx][:17], dtype=np.float32)
        valid = body_sc >= kpt_thr
        if not np.any(valid):
            continue
        pts = body_pts[valid]
        area = float((pts[:, 0].max() - pts[:, 0].min()) * (pts[:, 1].max() - pts[:, 1].min()))
        weight = float(body_sc[valid].mean()) * (area ** 0.5 + 1.0)
        if weight > best_weight:
            best_weight, best_idx = weight, idx
    return np.asarray(kpts[best_idx], dtype=np.float32), np.asarray(scores[best_idx], dtype=np.float32)


def _point_angle_deg(a, b, c):
    v1, v2 = a - b, c - b
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return None
    return float(np.degrees(np.arccos(np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0))))


def _line_tilt_deg(p_left, p_right):
    dx, dy = float(p_left[0] - p_right[0]), float(p_left[1] - p_right[1])
    if abs(dx) < 1e-6 and abs(dy) < 1e-6:
        return None
    return float(np.degrees(np.arctan2(dy, dx)))


def extract_pose_variables(kpts, scores, img_shape, kpt_thr=0.35):
    h, w = img_shape[:2]
    p, s = _select_primary_person(kpts, scores, kpt_thr=kpt_thr)
    if p is None or s is None:
        return {
            "detection_quality": {
                "is_valid": False,
                "person_count": 0,
                "body_valid_ratio": 0.0,
                "body_mean_score": 0.0,
            },
            "framing": {},
            "orientation": {},
            "proportions": {},
            "joint_angles": {},
            "normalized_keypoints": {},
        }

    body_p, body_s = p[:17], s[:17]
    has = lambda *names: all(body_s[KP[n]] >= kpt_thr for n in names)
    pt = lambda name: body_p[KP[name]]

    valid_mask = body_s >= kpt_thr
    valid_pts = body_p[valid_mask]

    bbox_w_ratio = bbox_h_ratio = center_x = center_y = None
    if len(valid_pts) > 0:
        bbox_w_ratio = (valid_pts[:, 0].max() - valid_pts[:, 0].min()) / w
        bbox_h_ratio = (valid_pts[:, 1].max() - valid_pts[:, 1].min()) / h

    if has("left_shoulder", "right_shoulder", "left_hip", "right_hip"):
        core = np.mean([pt("left_shoulder"), pt("right_shoulder"), pt("left_hip"), pt("right_hip")], axis=0)
        center_x, center_y = core[0] / w, core[1] / h
    elif has("left_shoulder", "right_shoulder"):
        core = np.mean([pt("left_shoulder"), pt("right_shoulder")], axis=0)
        center_x, center_y = core[0] / w, core[1] / h
    elif len(valid_pts) > 0:
        core = np.mean(valid_pts, axis=0)
        center_x, center_y = core[0] / w, core[1] / h

    shoulder_width = float(np.linalg.norm(pt("left_shoulder") - pt("right_shoulder"))) if has("left_shoulder", "right_shoulder") else None
    eye_dist = float(np.linalg.norm(pt("left_eye") - pt("right_eye"))) if has("left_eye", "right_eye") else None

    if has("left_ankle") or has("right_ankle"):
        estimated_shot = "full_body"
    elif has("left_knee") or has("right_knee"):
        estimated_shot = "cowboy_shot"
    elif has("left_hip") or has("right_hip"):
        estimated_shot = "upper_body"
    elif has("left_shoulder") or has("right_shoulder"):
        estimated_shot = "bust_up"
    else:
        estimated_shot = "close_up"

    head_roll_deg = _line_tilt_deg(pt("left_eye"), pt("right_eye")) if has("left_eye", "right_eye") else None
    shoulder_tilt_deg = _line_tilt_deg(pt("left_shoulder"), pt("right_shoulder")) if has("left_shoulder", "right_shoulder") else None

    head_yaw_proxy = None
    if has("nose", "left_eye", "right_eye") and eye_dist and eye_dist > 1e-3:
        eye_mid = 0.5 * (pt("left_eye") + pt("right_eye"))
        eye_vec = pt("left_eye") - pt("right_eye")
        eye_unit = eye_vec / np.linalg.norm(eye_vec)
        head_yaw_proxy = float(np.clip(float(np.dot(pt("nose") - eye_mid, eye_unit)) / (eye_dist * 0.5), -2.0, 2.0))

    torso_tilt_deg = torso_len = None
    if has("left_shoulder", "right_shoulder", "left_hip", "right_hip"):
        torso_vec = 0.5 * (pt("left_hip") + pt("right_hip")) - 0.5 * (pt("left_shoulder") + pt("right_shoulder"))
        torso_len = float(np.linalg.norm(torso_vec))
        if torso_len > 1e-3:
            torso_tilt_deg = float(np.degrees(np.arctan2(torso_vec[0], torso_vec[1])))

    if head_yaw_proxy is None:
        facing_direction = "unknown"
    elif abs(head_yaw_proxy) < 0.25:
        facing_direction = "front"
    elif abs(head_yaw_proxy) < 0.75:
        facing_direction = "three_quarter_right" if head_yaw_proxy > 0 else "three_quarter_left"
    else:
        facing_direction = "side_right" if head_yaw_proxy > 0 else "side_left"

    eye_to_shoulder = torso_to_shoulder = hip_to_shoulder = upper_arm_to_shoulder = None
    if shoulder_width and shoulder_width > 1e-3:
        if eye_dist is not None:
            eye_to_shoulder = eye_dist / shoulder_width
        if torso_len is not None:
            torso_to_shoulder = torso_len / shoulder_width
        if has("left_hip", "right_hip"):
            hip_to_shoulder = float(np.linalg.norm(pt("left_hip") - pt("right_hip"))) / shoulder_width
        arm_lens = []
        if has("left_shoulder", "left_elbow"):
            arm_lens.append(float(np.linalg.norm(pt("left_elbow") - pt("left_shoulder"))))
        if has("right_shoulder", "right_elbow"):
            arm_lens.append(float(np.linalg.norm(pt("right_elbow") - pt("right_shoulder"))))
        if arm_lens:
            upper_arm_to_shoulder = float(np.mean(arm_lens)) / shoulder_width

    left_elbow_deg = _point_angle_deg(pt("left_shoulder"), pt("left_elbow"), pt("left_wrist")) if has("left_shoulder", "left_elbow", "left_wrist") else None
    right_elbow_deg = _point_angle_deg(pt("right_shoulder"), pt("right_elbow"), pt("right_wrist")) if has("right_shoulder", "right_elbow", "right_wrist") else None
    left_knee_deg = _point_angle_deg(pt("left_hip"), pt("left_knee"), pt("left_ankle")) if has("left_hip", "left_knee", "left_ankle") else None
    right_knee_deg = _point_angle_deg(pt("right_hip"), pt("right_knee"), pt("right_ankle")) if has("right_hip", "right_knee", "right_ankle") else None

    normalized_keypoints = {}
    if has("left_shoulder", "right_shoulder") and shoulder_width and shoulder_width > 1e-3:
        origin = 0.5 * (pt("left_shoulder") + pt("right_shoulder"))
        for idx, name in enumerate(COCO_BODY_NAMES):
            rel = (body_p[idx] - origin) / shoulder_width
            normalized_keypoints[name] = {
                "x": _round_or_none(rel[0], 4),
                "y": _round_or_none(rel[1], 4),
                "score": _round_or_none(body_s[idx], 4),
            }

    return {
        "detection_quality": {
            "is_valid": bool(np.sum(valid_mask) >= 4),
            "person_count": int(len(kpts)),
            "body_valid_ratio": _round_or_none(float(np.mean(valid_mask)), 4),
            "body_mean_score": _round_or_none(float(np.mean(body_s)), 4),
        },
        "framing": {
            "estimated_shot": estimated_shot,
            "bbox_w_ratio": _round_or_none(bbox_w_ratio),
            "bbox_h_ratio": _round_or_none(bbox_h_ratio),
            "center_x": _round_or_none(center_x),
            "center_y": _round_or_none(center_y),
            "shoulder_width_ratio": _round_or_none((shoulder_width / w) if shoulder_width else None),
            "eye_dist_ratio": _round_or_none((eye_dist / w) if eye_dist else None),
        },
        "orientation": {
            "facing_direction": facing_direction,
            "head_yaw_proxy": _round_or_none(head_yaw_proxy),
            "head_roll_deg": _round_or_none(head_roll_deg, 2),
            "shoulder_tilt_deg": _round_or_none(shoulder_tilt_deg, 2),
            "torso_tilt_deg": _round_or_none(torso_tilt_deg, 2),
        },
        "proportions": {
            "eye_to_shoulder_ratio": _round_or_none(eye_to_shoulder),
            "torso_to_shoulder_ratio": _round_or_none(torso_to_shoulder),
            "hip_to_shoulder_ratio": _round_or_none(hip_to_shoulder),
            "upper_arm_to_shoulder_ratio": _round_or_none(upper_arm_to_shoulder),
        },
        "joint_angles": {
            "left_elbow_deg": _round_or_none(left_elbow_deg, 2),
            "right_elbow_deg": _round_or_none(right_elbow_deg, 2),
            "left_knee_deg": _round_or_none(left_knee_deg, 2),
            "right_knee_deg": _round_or_none(right_knee_deg, 2),
        },
        "normalized_keypoints": normalized_keypoints,
    }


def extract_tag_variables(wd14_pred: dict, slot_definitions=VARIABLE_TAG_SLOTS, general_threshold=0.35):
    general_scores = wd14_pred["general"]
    slots, slot_dominant = {}, {}
    for slot_name, tag_list in slot_definitions.items():
        slot_dict = {tag: round(float(general_scores.get(tag, 0.0)), 4) for tag in tag_list}
        slots[slot_name] = slot_dict
        if slot_dict:
            top_tag, top_score = max(slot_dict.items(), key=lambda kv: kv[1])
            slot_dominant[slot_name] = {"tag": top_tag, "score": top_score}

    active_general = {
        tag: round(float(score), 4)
        for tag, score in sorted(general_scores.items(), key=lambda kv: kv[1], reverse=True)
        if score >= general_threshold
    }
    return {
        "slots": slots,
        "slot_dominant": slot_dominant,
        "rating": {k: round(float(v), 4) for k, v in wd14_pred["rating"].items()},
        "active_general_tags": active_general,
    }

In [ ]:
# ==========================================
# 一括計測（WD14 + rtmlib + DINOv2）& 探索関数
# ==========================================
def analyze_candidates(
    candidate_items: list,
    pose_model=None,
    tagger=None,
    embedder=None,
    anchor_paths: list = None,
    slot_definitions: dict = VARIABLE_TAG_SLOTS,
    kpt_thr: float = 0.35,
    general_threshold: float = 0.35,
    tag_keep_min_max: float = 0.15,
):
    """
    全候補画像に対して rtmlib / WD14-Tagger / DINOv2 を一括実行し、
    評価値テーブル（metrics_df）・タグ信頼度行列（tag_matrix_df）・類似度行列（sim_matrix）を構築する。
    """
    pose_model = pose_model or wholebody
    tagger = tagger or wd14_tagger
    embedder = embedder or dinov2_embedder

    assert len(candidate_items) > 0, "候補画像が1枚も見つかりません。"

    records = []
    general_preds = []
    embeddings = []

    for idx, item in enumerate(candidate_items, 1):
        img_bgr = cv2.imread(item["file_path"], cv2.IMREAD_UNCHANGED)
        assert img_bgr is not None, f"画像が読み込めません: {item['file_path']}"

        if img_bgr.ndim == 2:
            bgr_for_pose = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
        elif img_bgr.shape[2] == 4:
            alpha = img_bgr[:, :, 3:4].astype(np.float32) / 255.0
            bgr_for_pose = (img_bgr[:, :, :3].astype(np.float32) * alpha + 255.0 * (1.0 - alpha)).astype(np.uint8)
        else:
            bgr_for_pose = img_bgr

        h, w = bgr_for_pose.shape[:2]
        kpts, scores = pose_model(bgr_for_pose)
        pose_vars = extract_pose_variables(kpts, scores, bgr_for_pose.shape, kpt_thr=kpt_thr)
        wd14_pred = tagger.predict(img_bgr)
        tag_vars = extract_tag_variables(wd14_pred, slot_definitions=slot_definitions, general_threshold=general_threshold)
        emb = embedder.embed(img_bgr)

        records.append({
            **item,
            "meta": {"width": int(w), "height": int(h), "aspect_ratio": round(w / h, 4)},
            "pose": pose_vars,
            "tags": tag_vars,
        })
        general_preds.append(wd14_pred["general"])
        embeddings.append(emb)

        if idx % 10 == 0 or idx == len(candidate_items):
            print(f"[Analyze {idx}/{len(candidate_items)}] latest={item['image_id']}")

    # --- DINOv2 コサイン類似度指標の算出 ---
    emb_mat = np.vstack(embeddings).astype(np.float32)  # (N, D), L2正規化済み
    sim_matrix = np.clip(emb_mat @ emb_mat.T, -1.0, 1.0)
    n = len(records)

    # 全体重心との類似度
    global_centroid = emb_mat.mean(axis=0)
    global_centroid /= max(float(np.linalg.norm(global_centroid)), 1e-8)
    global_centroid_sims = emb_mat @ global_centroid

    # パターン内重心との類似度
    patterns = [r["pattern"] for r in records]
    pattern_centroid_sims = np.zeros(n, dtype=np.float32)
    for pat in sorted(set(patterns)):
        idxs = [i for i, p in enumerate(patterns) if p == pat]
        pat_c = emb_mat[idxs].mean(axis=0)
        pat_c /= max(float(np.linalg.norm(pat_c)), 1e-8)
        pattern_centroid_sims[idxs] = emb_mat[idxs] @ pat_c

    # アンカー画像（任意）との類似度
    anchor_sims = None
    if anchor_paths:
        anchor_embs = []
        for ap in anchor_paths:
            a_img = cv2.imread(ap, cv2.IMREAD_UNCHANGED)
            if a_img is not None:
                anchor_embs.append(embedder.embed(a_img))
        if anchor_embs:
            anchor_c = np.vstack(anchor_embs).mean(axis=0)
            anchor_c /= max(float(np.linalg.norm(anchor_c)), 1e-8)
            anchor_sims = emb_mat @ anchor_c

    # 自分以外の最類似画像（重複検知用）
    sim_no_diag = sim_matrix.copy()
    np.fill_diagonal(sim_no_diag, -1.0)
    nearest_idxs = np.argmax(sim_no_diag, axis=1)
    max_pair_sims = sim_no_diag[np.arange(n), nearest_idxs] if n > 1 else np.zeros(n)

    # --- metrics_df の構築 ---
    rows = []
    for i, rec in enumerate(records):
        pose = rec["pose"]
        dq = pose["detection_quality"]
        fr = pose["framing"]
        ori = pose["orientation"]
        pr = pose["proportions"]
        dom = rec["tags"]["slot_dominant"]

        row = {
            "image_id": rec["image_id"],
            "pattern": rec["pattern"],
            "file_name": rec["file_name"],
            "file_path": rec["file_path"],
            "width": rec["meta"]["width"],
            "height": rec["meta"]["height"],
            # DINOv2 視覚的一貫性・重複指標
            "visual_centroid_sim": round(float(global_centroid_sims[i]), 4),
            "pattern_centroid_sim": round(float(pattern_centroid_sims[i]), 4),
            "max_pair_sim": round(float(max_pair_sims[i]), 4),
            "nearest_image": records[int(nearest_idxs[i])]["image_id"] if n > 1 else None,
            # rtmlib 骨格品質・構図・プロポーション
            "pose_valid": dq["is_valid"],
            "person_count": dq["person_count"],
            "body_mean_score": dq["body_mean_score"],
            "body_valid_ratio": dq["body_valid_ratio"],
            "estimated_shot": fr.get("estimated_shot"),
            "facing_direction": ori.get("facing_direction"),
            "head_yaw_proxy": ori.get("head_yaw_proxy"),
            "head_roll_deg": ori.get("head_roll_deg"),
            "shoulder_tilt_deg": ori.get("shoulder_tilt_deg"),
            "center_x": fr.get("center_x"),
            "center_y": fr.get("center_y"),
            "eye_to_shoulder_ratio": pr.get("eye_to_shoulder_ratio"),
            "torso_to_shoulder_ratio": pr.get("torso_to_shoulder_ratio"),
            "hip_to_shoulder_ratio": pr.get("hip_to_shoulder_ratio"),
            "upper_arm_to_shoulder_ratio": pr.get("upper_arm_to_shoulder_ratio"),
            # WD14 スロット優勢タグ
            "top_expression": dom.get("expression", {}).get("tag"),
            "top_expression_score": dom.get("expression", {}).get("score"),
            "top_gaze": dom.get("gaze", {}).get("tag"),
            "top_gaze_score": dom.get("gaze", {}).get("score"),
            "top_framing_tag": dom.get("framing", {}).get("tag"),
            "top_framing_score": dom.get("framing", {}).get("score"),
            "top_bg_tag": dom.get("background", {}).get("tag"),
            "top_bg_score": dom.get("background", {}).get("score"),
        }
        if anchor_sims is not None:
            row["anchor_sim"] = round(float(anchor_sims[i]), 4)
        rows.append(row)

    metrics_df = pd.DataFrame(rows)

    # --- タグ信頼度行列（いずれかの画像で tag_keep_min_max 以上になったタグを保持） ---
    all_tags = list(general_preds[0].keys())
    full_tag_mat = np.array([[p[t] for t in all_tags] for p in general_preds], dtype=np.float32)
    keep_mask = np.max(full_tag_mat, axis=0) >= tag_keep_min_max
    kept_tags = [t for t, k in zip(all_tags, keep_mask) if k]
    tag_matrix_df = pd.DataFrame(
        full_tag_mat[:, keep_mask],
        index=metrics_df["image_id"].tolist(),
        columns=kept_tags,
    )

    return records, metrics_df, tag_matrix_df, sim_matrix, general_preds


def summarize_tag_landscape(tag_matrix_df: pd.DataFrame, metrics_df: pd.DataFrame, active_thr: float = 0.35) -> pd.DataFrame:
    """
    全候補画像における WD14 タグの安定度（平均・標準偏差・出現率・パターン間差）を集計する。
    """
    variable_tags = {t for tags in VARIABLE_TAG_SLOTS.values() for t in tags}
    pat_series = metrics_df.set_index("image_id")["pattern"]
    pat_means = tag_matrix_df.groupby(pat_series).mean()

    rows = []
    for tag in tag_matrix_df.columns:
        s = tag_matrix_df[tag]
        role = "variable_slot" if tag in variable_tags else ("ignored" if tag in IGNORED_IDENTITY_TAGS else "trait_candidate")
        rows.append({
            "tag": tag,
            "role": role,
            "mean": round(float(s.mean()), 4),
            "std": round(float(s.std(ddof=0)), 4),
            "presence_rate": round(float((s >= active_thr).mean()), 4),
            "pattern_mean_std": round(float(pat_means[tag].std(ddof=0)), 4) if len(pat_means) > 1 else 0.0,
            "min": round(float(s.min()), 4),
            "max": round(float(s.max()), 4),
        })
    return pd.DataFrame(rows).sort_values(["mean", "presence_rate"], ascending=False).reset_index(drop=True)

In [ ]:
# ==========================================
# 探索用可視化 & 選定（Curate）関数
# ==========================================
def show_exploration_overview(metrics_df: pd.DataFrame, tag_landscape_df: pd.DataFrame, figsize=(15, 10)):
    """
    学習候補データの全体像を4パネルで可視化する（DINOv2一貫性/重複・骨格品質・頭身比・WD14タグ安定度）。
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    patterns = sorted(metrics_df["pattern"].unique())
    cmap = plt.get_cmap("tab10")

    # 1. DINOv2: visual_centroid_sim vs max_pair_sim
    ax = axes[0, 0]
    for idx, pat in enumerate(patterns):
        sub = metrics_df[metrics_df["pattern"] == pat]
        ax.scatter(
            sub["visual_centroid_sim"],
            sub["max_pair_sim"],
            label=f"{pat} (n={len(sub)})",
            color=cmap(idx % 10),
            alpha=0.8,
            s=45,
        )
    ax.set_xlabel("DINOv2 visual_centroid_sim (Higher = More Consistent)")
    ax.set_ylabel("DINOv2 max_pair_sim (Higher = Near Duplicate)")
    ax.set_title("1. Visual Consistency vs Duplicate Risk (DINOv2)")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.35)

    # 2. rtmlib: body_mean_score パターン別分布
    ax = axes[0, 1]
    data_scores = [metrics_df[metrics_df["pattern"] == p]["body_mean_score"].dropna().values for p in patterns]
    ax.boxplot(data_scores, tick_labels=patterns, patch_artist=True)
    ax.set_ylabel("rtmlib body_mean_score")
    ax.set_title("2. Pose Detection Quality by Pattern")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    # 3. rtmlib: eye_to_shoulder_ratio パターン別分布
    ax = axes[1, 0]
    data_prop = [metrics_df[metrics_df["pattern"] == p]["eye_to_shoulder_ratio"].dropna().values for p in patterns]
    if any(len(d) > 0 for d in data_prop):
        ax.boxplot(data_prop, tick_labels=patterns, patch_artist=True)
        ax.set_ylabel("eye_to_shoulder_ratio")
        ax.set_title("3. Proportion (Eye / Shoulder Ratio) by Pattern")
        ax.tick_params(axis="x", rotation=25)
        ax.grid(axis="y", linestyle="--", alpha=0.35)
    else:
        ax.text(0.5, 0.5, "No Valid Proportion Data", ha="center", va="center")
        ax.set_axis_off()

    # 4. WD14: タグ平均 vs 標準偏差（安定固有特徴 vs 可変特徴マップ）
    ax = axes[1, 1]
    sub_tags = tag_landscape_df[tag_landscape_df["mean"] >= 0.15]
    role_colors = {"trait_candidate": "#4C78A8", "variable_slot": "#F58518", "ignored": "#999999"}
    for role, color in role_colors.items():
        r_df = sub_tags[sub_tags["role"] == role]
        ax.scatter(r_df["mean"], r_df["std"], label=role, color=color, alpha=0.75, s=35)
    # 特徴的なタグを上位いくつかラベル付け
    for _, row in sub_tags.head(12).iterrows():
        ax.annotate(row["tag"], (row["mean"], row["std"]), fontsize=7.5, xytext=(4, 2), textcoords="offset points")
    ax.set_xlabel("WD14 Mean Confidence")
    ax.set_ylabel("WD14 Std (Lower = More Stable)")
    ax.set_title("4. WD14 Tag Stability Map (Right-Bottom = Stable Core Traits)")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.35)

    plt.tight_layout()
    plt.show()


def inspect_tag_across_patterns(tag_matrix_df: pd.DataFrame, metrics_df: pd.DataFrame, tag_names: list, figsize=(13, 4)):
    """任意の WD14 タグ群について、パターン別の信頼度分布（平均・標準偏差・箱ひげ）を比較する。"""
    valid_tags = [t for t in tag_names if t in tag_matrix_df.columns]
    missing = [t for t in tag_names if t not in tag_matrix_df.columns]
    if missing:
        print(f"ℹ️ 候補画像にほとんど出現していないタグ（スキップ）: {missing}")
    if not valid_tags:
        return

    merged = tag_matrix_df[valid_tags].copy()
    merged["pattern"] = metrics_df.set_index("image_id")["pattern"]
    summary = merged.groupby("pattern")[valid_tags].agg(["mean", "std", "min", "max"]).round(3)
    display(summary)

    patterns = sorted(merged["pattern"].unique())
    means = merged.groupby("pattern")[valid_tags].mean()
    ax = means.plot(kind="bar", figsize=figsize, ylim=(0.0, 1.05), width=0.8)
    ax.set_ylabel("Mean WD14 Confidence")
    ax.set_title("Selected WD14 Tags by Pattern")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()


def show_extreme_thumbnails(metrics_df: pd.DataFrame, column: str, ascending: bool = True, n: int = 8, cols: int = 4, extra_cols: list = None):
    """
    任意の評価値カラムでソートした上位/下位 N 枚のサムネイルを値付きで一覧表示する。
    しきい値を決める際の目視確認に使う。
    """
    sub = metrics_df.dropna(subset=[column]).sort_values(column, ascending=ascending).head(n)
    if len(sub) == 0:
        print(f"ℹ️ カラム '{column}' に有効な値がありません。")
        return

    rows_n = math.ceil(len(sub) / cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.6, rows_n * 4.0))
    axes = np.atleast_1d(axes).ravel()
    direction = "Lowest" if ascending else "Highest"
    fig.suptitle(f"{direction} {len(sub)} by '{column}'", fontsize=12)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, sub.iterrows()):
        img = cv2.imread(row["file_path"])
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        lines = [f"{row['image_id']}", f"{column}={row[column]:.3f}"]
        for ec in (extra_cols or []):
            if ec in row and pd.notna(row[ec]):
                val = f"{row[ec]:.3f}" if isinstance(row[ec], float) else str(row[ec])
                lines.append(f"{ec}={val}")
        ax.set_title("\n".join(lines), fontsize=8.5)
        ax.axis("on")
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()


def apply_selection_rules(
    metrics_df: pd.DataFrame,
    tag_matrix_df: pd.DataFrame,
    sim_matrix: np.ndarray,
    rules: dict,
) -> pd.DataFrame:
    """
    探索結果から決めた選定ルール（rules）を全候補画像に適用し、
    selected (bool) と reject_reasons (str) を付与した DataFrame を返す。
    """
    df = metrics_df.copy().reset_index(drop=True)
    reasons_list = [[] for _ in range(len(df))]

    # 骨格プロポーション外れ値用の統計（全有効データ基準）
    prop_z_max = rules.get("proportion_z_max")
    prop_stats = {}
    if prop_z_max is not None:
        for col in ["eye_to_shoulder_ratio", "torso_to_shoulder_ratio", "hip_to_shoulder_ratio"]:
            vals = df[col].dropna()
            if len(vals) >= 3:
                prop_stats[col] = (float(vals.mean()), max(float(vals.std(ddof=0)), 0.04))

    for i, row in df.iterrows():
        img_id = row["image_id"]
        pat = row["pattern"]
        r = reasons_list[i]

        # 1. DINOv2 視覚的一貫性
        min_v_sim = rules.get("min_visual_centroid_sim")
        if min_v_sim is not None and row["visual_centroid_sim"] < min_v_sim:
            r.append(f"low_visual_sim({row['visual_centroid_sim']:.3f}<{min_v_sim})")

        min_p_sim = rules.get("min_pattern_centroid_sim")
        if min_p_sim is not None and row["pattern_centroid_sim"] < min_p_sim:
            r.append(f"low_pattern_sim({row['pattern_centroid_sim']:.3f}<{min_p_sim})")

        min_a_sim = rules.get("min_anchor_sim")
        if min_a_sim is not None and "anchor_sim" in row and pd.notna(row["anchor_sim"]) and row["anchor_sim"] < min_a_sim:
            r.append(f"low_anchor_sim({row['anchor_sim']:.3f}<{min_a_sim})")

        # 2. rtmlib 骨格品質・人数・プロポーション
        if rules.get("require_pose_valid", False) and not bool(row["pose_valid"]):
            r.append("pose_invalid")

        max_persons = rules.get("max_person_count")
        if max_persons is not None and row["person_count"] > max_persons:
            r.append(f"person_count({row['person_count']}>{max_persons})")

        min_pose_score = rules.get("min_body_mean_score")
        if min_pose_score is not None and row["body_mean_score"] < min_pose_score:
            r.append(f"low_pose_score({row['body_mean_score']:.3f}<{min_pose_score})")

        if prop_z_max is not None:
            for col, (m, sd) in prop_stats.items():
                if pd.notna(row[col]):
                    z = abs((float(row[col]) - m) / sd)
                    if z > prop_z_max:
                        r.append(f"prop_outlier:{col}(z={z:.2f})")

        # 3. WD14 必須タグ / 禁止タグ（全体 + パターン別上書き）
        req_tags = dict(rules.get("required_tags", {}))
        forb_tags = dict(rules.get("forbidden_tags", {}))
        pat_cfg = rules.get("pattern_rules", {}).get(pat, {})
        req_tags.update(pat_cfg.get("required_tags", {}))
        forb_tags.update(pat_cfg.get("forbidden_tags", {}))

        for tag, min_sc in req_tags.items():
            actual = float(tag_matrix_df.loc[img_id, tag]) if tag in tag_matrix_df.columns else 0.0
            if actual < min_sc:
                r.append(f"missing_tag:{tag}({actual:.2f}<{min_sc})")

        for tag, max_sc in forb_tags.items():
            actual = float(tag_matrix_df.loc[img_id, tag]) if tag in tag_matrix_df.columns else 0.0
            if actual > max_sc:
                r.append(f"forbidden_tag:{tag}({actual:.2f}>{max_sc})")

    # 4. DINOv2 重複除外（現時点で合格した画像同士から visual_centroid_sim が高い方を優先保持）
    dedup_thr = rules.get("max_pair_sim_dedup")
    if dedup_thr is not None:
        pass_indices = [i for i in range(len(df)) if len(reasons_list[i]) == 0]
        pass_indices.sort(key=lambda i: float(df.loc[i, "visual_centroid_sim"]), reverse=True)
        kept = []
        for idx in pass_indices:
            dup_of = None
            for k in kept:
                if sim_matrix[idx, k] >= dedup_thr:
                    dup_of = (df.loc[k, "image_id"], float(sim_matrix[idx, k]))
                    break
            if dup_of is not None:
                reasons_list[idx].append(f"near_duplicate:{dup_of[0]}({dup_of[1]:.3f}>={dedup_thr})")
            else:
                kept.append(idx)

    df["selected"] = [len(r) == 0 for r in reasons_list]
    df["reject_reasons"] = ["; ".join(r) for r in reasons_list]
    return df


def show_selection_summary(curated_df: pd.DataFrame):
    """採用/除外の集計・パターン別内訳・ショット/向きバランス・主な除外理由を表示する。"""
    total = len(curated_df)
    n_sel = int(curated_df["selected"].sum())
    print(f"=== 選定サマリー: 採用 {n_sel} / {total} 枚 ({(n_sel / max(total, 1)) * 100:.1f}%) ===")

    pat_summary = (
        curated_df.groupby("pattern")["selected"]
        .agg(total="count", selected="sum")
        .assign(rejected=lambda d: d["total"] - d["selected"], accept_rate=lambda d: (d["selected"] / d["total"]).round(3))
    )
    display(pat_summary)

    sel_df = curated_df[curated_df["selected"]]
    if len(sel_df) > 0:
        print("\n--- 採用データのショット種別 × パターン分布 ---")
        display(pd.crosstab(sel_df["pattern"], sel_df["estimated_shot"].fillna("unknown"), margins=True))
        print("\n--- 採用データの顔の向き × パターン分布 ---")
        display(pd.crosstab(sel_df["pattern"], sel_df["facing_direction"].fillna("unknown"), margins=True))

    rej_df = curated_df[~curated_df["selected"]]
    if len(rej_df) > 0:
        all_reasons = []
        for s in rej_df["reject_reasons"]:
            for part in s.split("; "):
                all_reasons.append(part.split("(")[0])
        print("\n--- 主な除外理由（延べ件数） ---")
        display(pd.Series(all_reasons).value_counts().to_frame("count"))


def show_curated_thumbnails(curated_df: pd.DataFrame, selected: bool = False, n: int = 8, cols: int = 4):
    """採用または除外された画像のサムネイルを確認する（除外時は除外理由を表示）。"""
    sub = curated_df[curated_df["selected"] == selected].head(n)
    label = "Selected" if selected else "Rejected"
    if len(sub) == 0:
        print(f"ℹ️ {label} に該当する画像はありません。")
        return

    rows_n = math.ceil(len(sub) / cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.8, rows_n * 4.2))
    axes = np.atleast_1d(axes).ravel()
    fig.suptitle(f"{label} Samples (showing {len(sub)})", fontsize=12)
    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, sub.iterrows()):
        img = cv2.imread(row["file_path"])
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        title = f"{row['image_id']}\nv_sim={row['visual_centroid_sim']:.3f} | pose={row['body_mean_score']:.2f}"
        if not selected and row["reject_reasons"]:
            title += f"\n[{row['reject_reasons'][:48]}]"
        ax.set_title(title, fontsize=8, color="#1B5E20" if selected else "#B71C1C")
        ax.axis("on")
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()


def export_selected_dataset(curated_df: pd.DataFrame, export_dir: str, preserve_pattern_dirs: bool = True):
    """採用画像を書き出し先フォルダへコピーする。"""
    os.makedirs(export_dir, exist_ok=True)
    sel_df = curated_df[curated_df["selected"]]
    for _, row in sel_df.iterrows():
        dst_dir = os.path.join(export_dir, row["pattern"]) if (preserve_pattern_dirs and row["pattern"] != "root") else export_dir
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(row["file_path"], os.path.join(dst_dir, row["file_name"]))
    print(f"✅ 採用画像 {len(sel_df)} 枚を書き出しました -> {export_dir}")


def copy_files_to_pattern(
    file_list_source,
    dest_root: str = "./data/candidates",
    target_pattern: str = None,
    pattern_col: str = None,
    source_dir: str = None,
    only_selected: bool = True,
    clear_dest: bool = False,
    dry_run: bool = False,
):
    """
    指定した CSV（パス / DataFrame）またはファイル名リストをもとに、
    画像を `dest_root/<pattern>/` へコピーする。
    - 固定パターンへまとめる場合: `target_pattern="pattern_a"` を指定
    - CSV の列ごとに振り分ける場合: `pattern_col="estimated_shot"` 等を指定
    """
    assert target_pattern or pattern_col, "target_pattern または pattern_col のどちらかを指定してください。"

    if isinstance(file_list_source, (str, Path)):
        df = pd.read_csv(file_list_source)
    elif isinstance(file_list_source, pd.DataFrame):
        df = file_list_source.copy()
    elif isinstance(file_list_source, (list, tuple)):
        df = pd.DataFrame({"file_path": [str(x) for x in file_list_source]})
    else:
        raise TypeError("file_list_source には CSVパス / DataFrame / ファイルリストを指定してください。")

    if only_selected and "selected" in df.columns:
        df = df[df["selected"].astype(str).str.lower().isin(["true", "1"])].copy()

    # コピー計画 (src_path, dst_pattern, dst_name) を作成
    plan = []
    missing = []
    for _, row in df.iterrows():
        src_path = None
        if "file_path" in df.columns and pd.notna(row.get("file_path")):
            cand = str(row["file_path"])
            if os.path.exists(cand):
                src_path = cand
            elif source_dir and os.path.exists(os.path.join(source_dir, os.path.basename(cand))):
                src_path = os.path.join(source_dir, os.path.basename(cand))
        if src_path is None and "file_name" in df.columns and pd.notna(row.get("file_name")):
            fname = str(row["file_name"])
            cand = os.path.join(source_dir, fname) if source_dir else fname
            if os.path.exists(cand):
                src_path = cand

        if src_path is None:
            missing.append(str(row.get("file_path") or row.get("file_name")))
            continue

        pat_name = str(target_pattern) if target_pattern else str(row.get(pattern_col, "unknown") or "unknown")
        pat_name = pat_name.strip().replace("/", "_").replace("\\", "_")
        plan.append((src_path, pat_name, os.path.basename(src_path)))

    if missing:
        print(f"⚠️ 見つからなかったファイル: {len(missing)} 件（例: {missing[:3]}）")

    pat_counts = pd.Series([p for _, p, _ in plan]).value_counts().to_dict() if plan else {}
    if dry_run:
        print(f"[DRY RUN] コピー予定: {len(plan)} 枚 -> {dest_root} | 内訳: {pat_counts}")
        for src, pat, name in plan[:5]:
            print(f"  {src} -> {os.path.join(dest_root, pat, name)}")
        return plan

    cleared = set()
    copied = 0
    for src, pat, name in plan:
        dst_dir = os.path.join(dest_root, pat)
        if clear_dest and dst_dir not in cleared:
            if os.path.exists(dst_dir):
                shutil.rmtree(dst_dir)
            cleared.add(dst_dir)
        os.makedirs(dst_dir, exist_ok=True)
        dst_path = os.path.join(dst_dir, name)
        if os.path.abspath(src) != os.path.abspath(dst_path):
            shutil.copy2(src, dst_path)
        copied += 1

    print(f"✅ コピー完了: {copied} 枚 -> {dest_root} | パターン内訳: {pat_counts}")
    return plan


def build_profile_from_selected(
    curated_df: pd.DataFrame,
    records: list,
    general_preds: list,
    character_name: str,
    core_presence_threshold: float = 0.65,
    core_mean_threshold: float = 0.45,
    tag_active_threshold: float = 0.35,
):
    """
    選定を通過した採用データ群から、snapshot-inspector 互換の character_profile.json を構築する。
    """
    sel_indices = curated_df.index[curated_df["selected"]].tolist()
    assert len(sel_indices) > 0, "採用画像が0枚のためプロファイルを作成できません。"

    sel_records = [records[i] for i in sel_indices]
    sel_preds = [general_preds[i] for i in sel_indices]
    variable_tag_set = {t for tags in VARIABLE_TAG_SLOTS.values() for t in tags}

    identity_core_tags = {}
    reference_general_mean = {}
    for tag in sel_preds[0].keys():
        vals = np.array([p[tag] for p in sel_preds], dtype=np.float64)
        mean_val = float(vals.mean())
        if float(vals.max()) >= 0.15:
            reference_general_mean[tag] = round(mean_val, 4)
        if tag in variable_tag_set or tag in IGNORED_IDENTITY_TAGS:
            continue
        pres = float((vals >= tag_active_threshold).mean())
        if pres >= core_presence_threshold and mean_val >= core_mean_threshold:
            identity_core_tags[tag] = {
                "mean": round(mean_val, 4),
                "std": round(float(vals.std(ddof=0)), 4),
                "median": round(float(np.median(vals)), 4),
                "min": round(float(vals.min()), 4),
                "max": round(float(vals.max()), 4),
                "presence_rate": round(pres, 4),
                "forced": False,
            }
    identity_core_tags = dict(sorted(identity_core_tags.items(), key=lambda kv: kv[1]["mean"], reverse=True))

    proportions = {}
    for key in ["eye_to_shoulder_ratio", "torso_to_shoulder_ratio", "hip_to_shoulder_ratio", "upper_arm_to_shoulder_ratio"]:
        vals = [r["pose"]["proportions"].get(key) for r in sel_records if r["pose"]["detection_quality"]["is_valid"]]
        clean = [float(v) for v in vals if v is not None]
        if clean:
            arr = np.asarray(clean, dtype=np.float64)
            proportions[key] = {
                "count": int(len(arr)),
                "mean": round(float(arr.mean()), 4),
                "std": round(float(arr.std(ddof=0)), 4),
                "median": round(float(np.median(arr)), 4),
                "min": round(float(arr.min()), 4),
                "max": round(float(arr.max()), 4),
            }

    variable_slots_baseline = {}
    for slot_name, tag_list in VARIABLE_TAG_SLOTS.items():
        slot_stats = {}
        for tag in tag_list:
            arr = np.asarray([r["tags"]["slots"][slot_name][tag] for r in sel_records], dtype=np.float64)
            slot_stats[tag] = {
                "count": int(len(arr)),
                "mean": round(float(arr.mean()), 4),
                "std": round(float(arr.std(ddof=0)), 4),
                "median": round(float(np.median(arr)), 4),
                "min": round(float(arr.min()), 4),
                "max": round(float(arr.max()), 4),
            }
        variable_slots_baseline[slot_name] = slot_stats

    return {
        "schema_version": "1.0",
        "character_name": character_name,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "wd14_model": WD14_MODEL_REPO,
        "dinov2_model": DINOV2_MODEL_ID,
        "reference_count": len(sel_records),
        "reference_files": [r["image_id"] for r in sel_records],
        "identity_core_tags": identity_core_tags,
        "proportions": proportions,
        "variable_slots_baseline": variable_slots_baseline,
        "reference_general_mean": reference_general_mean,
    }


def save_results_to_drive(output_dir: str, file_paths: list, prefix: str = ""):
    if not os.path.exists("/content/drive"):
        print("⚠️ 警告: '/content/drive' が見つかりません。先に Drive をマウントしてください。")
        return
    os.makedirs(output_dir, exist_ok=True)
    saved = 0
    for src in file_paths:
        if not os.path.exists(src):
            print(f"ℹ️ スキップ（未生成）: {src}")
            continue
        dst = os.path.join(output_dir, f"{prefix}{os.path.basename(src)}")
        shutil.copy2(src, dst)
        print(f"✅ 保存完了: {dst}")
        saved += 1
    print(f"合計 {saved} ファイルを Drive に保存しました -> {output_dir}")

## 5. 全候補画像の一括計測（WD14 + rtmlib + DINOv2）

`CANDIDATE_ROOT` 配下の全パターン画像を一括計測し、評価値テーブル（`metrics_df`）とタグ信頼度行列（`tag_matrix_df`）を作成します。  
※ 基準にしたい厳選アンカー画像がすでにある場合は `ANCHOR_PATHS` に指定すると `anchor_sim` も同時に算出されます。

In [ ]:
CHARACTER_NAME = "my_character"
CANDIDATE_ROOT = "./data/candidates"
ANCHOR_PATHS = []  # 任意: 理想の絵柄・顔の基準画像パスリスト（空なら全体重心のみ使用）

KPT_THR = 0.35
TAG_ACTIVE_THRESHOLD = 0.35

candidate_items = discover_candidate_images(CANDIDATE_ROOT)
print(f"検出された候補画像数: {len(candidate_items)}")
if candidate_items:
    pat_counts = pd.Series([x["pattern"] for x in candidate_items]).value_counts()
    display(pat_counts.to_frame("image_count"))

records, metrics_df, tag_matrix_df, sim_matrix, general_preds = analyze_candidates(
    candidate_items=candidate_items,
    anchor_paths=ANCHOR_PATHS or None,
    kpt_thr=KPT_THR,
    general_threshold=TAG_ACTIVE_THRESHOLD,
)

tag_landscape_df = summarize_tag_landscape(tag_matrix_df, metrics_df, active_thr=TAG_ACTIVE_THRESHOLD)

os.makedirs("./output", exist_ok=True)
metrics_df.to_csv("./output/candidate_metrics.csv", index=False, encoding="utf-8-sig")
tag_landscape_df.to_csv("./output/tag_landscape.csv", index=False, encoding="utf-8-sig")
print("✅ 計測結果を保存しました -> ./output/candidate_metrics.csv, ./output/tag_landscape.csv")

display(metrics_df.head(10))

## 6. 評価値の探索・分布確認（選定基準づくり）

計測した評価値の分布・タグ安定度・極端値サムネイルを確認しながら、**どの指標にどのしきい値を置くべきか**を探ります。

In [ ]:
# 1. 4パネル探索ダッシュボード（DINOv2一貫性/重複・骨格品質・頭身比・WD14タグ安定度）
show_exploration_overview(metrics_df, tag_landscape_df)

# 2. 候補全体で安定して出ている「キャラ固有特徴候補タグ」上位25件
print("=== 安定しているキャラ固有特徴候補タグ（上位25） ===")
display(tag_landscape_df[tag_landscape_df["role"] == "trait_candidate"].head(25))

# 3. パターン間で差が大きいタグ（パターン切り替えが効いているタグ / 意図せずブレているタグ）
print("=== パターン間で平均信頼度の差が大きいタグ（上位20） ===")
display(tag_landscape_df.sort_values("pattern_mean_std", ascending=False).head(20))

In [ ]:
# 4. 気になる任意の WD14 タグをパターン別に比較（ここを自由に書き換えて探索）
TAGS_TO_INSPECT = [
    "smile",
    "looking_at_viewer",
    "upper_body",
    "cowboy_shot",
    "full_body",
    "simple_background",
]
inspect_tag_across_patterns(tag_matrix_df, metrics_df, TAGS_TO_INSPECT)

In [ ]:
# 5. 極端値サムネイルの目視確認（しきい値の肌感覚を掴む）
# DINOv2 全体重心との類似度が低い画像（絵柄・顔つきがズレた外れ値候補）
show_extreme_thumbnails(
    metrics_df,
    column="visual_centroid_sim",
    ascending=True,
    n=8,
    extra_cols=["body_mean_score", "estimated_shot"],
)

# DINOv2 他画像との最大類似度が高い画像（重複・似すぎ候補）
show_extreme_thumbnails(
    metrics_df,
    column="max_pair_sim",
    ascending=False,
    n=8,
    extra_cols=["nearest_image", "visual_centroid_sim"],
)

# rtmlib 骨格検出スコアが低い画像（ポーズ破綻・顔崩れ候補）
show_extreme_thumbnails(
    metrics_df,
    column="body_mean_score",
    ascending=True,
    n=8,
    extra_cols=["visual_centroid_sim", "person_count"],
)

## 7. 選定ルールの適用・採用データ確認・書き出し

Section 6 の探索結果をもとに `SELECTION_RULES` を調整し、学習データとして採用する画像を選定します。  
納得いくまでこのセクションだけ何度でも再実行できます。

In [ ]:
# ==========================================
# 選定ルール設定（探索結果を見ながら調整）
# ==========================================
SELECTION_RULES = {
    # DINOv2 視覚的一貫性・重複除外（None にすると無効）
    "min_visual_centroid_sim": 0.65,   # 全体重心との類似度下限（絵柄・顔ブレ除外）
    "min_pattern_centroid_sim": None,  # パターン内重心との類似度下限
    "min_anchor_sim": None,            # ANCHOR_PATHS 指定時の下限
    "max_pair_sim_dedup": 0.96,        # これ以上似ているペアは視覚一貫性が高い方だけ残す

    # rtmlib 骨格・構図・プロポーション条件
    "require_pose_valid": True,        # 骨格検出が成立していること
    "max_person_count": 1,             # 複数人写り込みを除外
    "min_body_mean_score": 0.45,       # 骨格信頼度の下限
    "proportion_z_max": 2.8,           # 頭身プロポーションの外れ値（z-score上限）

    # WD14 全体必須タグ / 全体禁止タグ
    "required_tags": {
        # 例: "1girl": 0.70, "pointy_ears": 0.50
    },
    "forbidden_tags": {
        "monochrome": 0.50,
        "multiple_girls": 0.50,
    },

    # パターン別の個別ルール（任意）
    "pattern_rules": {
        # 例:
        # "pattern_a_upper": {"required_tags": {"upper_body": 0.50}},
        # "pattern_b_full": {"required_tags": {"full_body": 0.50}},
    },
}

curated_df = apply_selection_rules(metrics_df, tag_matrix_df, sim_matrix, SELECTION_RULES)
show_selection_summary(curated_df)

# 除外された画像と採用された画像のサムネイルを目視確認
show_curated_thumbnails(curated_df, selected=False, n=8)
show_curated_thumbnails(curated_df, selected=True, n=8)

In [ ]:
# ==========================================
# 採用データセットの書き出し & Character Profile 生成
# ==========================================
SELECTED_EXPORT_DIR = f"./output/{CHARACTER_NAME}_selected_dataset"
CURATED_CSV_PATH = f"./output/{CHARACTER_NAME}_curated_metrics.csv"
RULES_JSON_PATH = f"./output/{CHARACTER_NAME}_selection_rules.json"
PROFILE_JSON_PATH = f"./output/{CHARACTER_NAME}_character_profile.json"

export_selected_dataset(curated_df, SELECTED_EXPORT_DIR, preserve_pattern_dirs=True)
curated_df.to_csv(CURATED_CSV_PATH, index=False, encoding="utf-8-sig")

with open(RULES_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(SELECTION_RULES, f, ensure_ascii=False, indent=2)

# 採用データ群から snapshot-inspector 互換の Character Profile を構築
character_profile = build_profile_from_selected(
    curated_df=curated_df,
    records=records,
    general_preds=general_preds,
    character_name=CHARACTER_NAME,
    tag_active_threshold=TAG_ACTIVE_THRESHOLD,
)
with open(PROFILE_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(character_profile, f, ensure_ascii=False, indent=2)

print(f"✅ 選定結果 CSV      : {CURATED_CSV_PATH}")
print(f"✅ 選定ルール JSON   : {RULES_JSON_PATH}")
print(f"✅ Character Profile : {PROFILE_JSON_PATH} (core_tags={len(character_profile['identity_core_tags'])})")

### 7.5 指定ファイル一覧（CSV）から `pattern_*` フォルダへの再配置（次ループ準備）

内部で生成した CSV（`{CHARACTER_NAME}_curated_metrics.csv` や `candidate_metrics.csv`、手動編集した CSV）またはファイル名リストをもとに、画像を `pattern_*` サブフォルダへコピーします。  
コピー先を `./data/candidates`（別フォルダ `./data/candidates_grouped` でも可）にすれば、そのまま Section 5 を再実行してパターン別集計に移行できます。

In [ ]:
# ==========================================
# CSV / ファイル一覧から pattern_* フォルダへコピー
# ==========================================
# コピー先の親フォルダ（Section 5 の CANDIDATE_ROOT と同じにするか、新しい作業フォルダを指定）
RECOPY_DEST_ROOT = "./data/candidates_grouped"

# --- 使い方 A: 採用済み CSV を指定の固定パターンフォルダ（例: pattern_a）へコピー ---
copy_files_to_pattern(
    file_list_source=CURATED_CSV_PATH,   # CSVパス / curated_df / ["a.png", "b.png"] いずれも可
    dest_root=RECOPY_DEST_ROOT,
    target_pattern="pattern_a",          # コピー先の pattern_* フォルダ名
    only_selected=True,                  # selected==True の行だけコピー（全行なら False）
    clear_dest=False,                    # True にするとコピー先パターンフォルダを事前に空にする
    dry_run=True,                        # まず True で予定を確認し、問題なければ False にして実行
)

# --- 使い方 B: CSV 内の列（例: estimated_shot や facing_direction）ごとに自動振り分け ---
# copy_files_to_pattern(
#     file_list_source=CURATED_CSV_PATH,
#     dest_root=RECOPY_DEST_ROOT,
#     pattern_col="estimated_shot",      # 例: upper_body / cowboy_shot / full_body ごとに振り分け
#     only_selected=True,
#     clear_dest=True,
#     dry_run=False,
# )

# --- 使い方 C: ファイル名リスト（CSVに file_name 列だけある場合等）を source_dir からコピー ---
# copy_files_to_pattern(
#     file_list_source="./my_pattern_list.csv",  # file_name 列（または file_path 列）を持つ CSV
#     source_dir="./data/candidates",
#     dest_root=RECOPY_DEST_ROOT,
#     target_pattern="pattern_b",
#     only_selected=False,
#     dry_run=False,
# )

## 8. 結果のDrive保存・ランタイム終了メモ

In [ ]:
DRIVE_OUTPUT_DIR = f"/content/drive/MyDrive/Colab Notebooks/dataset-curation/{CHARACTER_NAME}"
OUTPUT_PREFIX = ""

TARGET_FILES = [
    "./output/candidate_metrics.csv",
    "./output/tag_landscape.csv",
    CURATED_CSV_PATH,
    RULES_JSON_PATH,
    PROFILE_JSON_PATH,
]
save_results_to_drive(DRIVE_OUTPUT_DIR, file_paths=TARGET_FILES, prefix=OUTPUT_PREFIX)

# 採用画像フォルダ自体も Drive にバックアップしたい場合は以下をコメント解除
# !cp -r {SELECTED_EXPORT_DIR} '{DRIVE_OUTPUT_DIR}/'

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する